# Classification Post-processing

This notebook post-processes the classified raster produced by the Maximum Likelihood Classification.

The post-processing consists of three main steps:

1. Extraction of the asbestos-cement class from the classified raster.
2. Noise reduction and removal of small isolated regions.
3. Aggregation of asbestos pixels at the building level to identify roofs exceeding a user-defined asbestos coverage threshold.

### Input

- Classified raster (`MLC_classification.tif`)
- Building footprint layer

### Output

- Binary asbestos mask (`Asbestos_mask.tif`)
- Cleaned asbestos mask (`Asbestos_mask_clean.tif`)
- Building layer containing the percentage of asbestos pixels
- Buildings exceeding the selected asbestos threshold

In [ ]:
from pathlib import Path

import cv2
import geopandas as gpd
import numpy as np
import rasterio

from rasterio.transform import xy
from shapely.geometry import Point
from skimage.morphology import remove_small_objects


# Input and output files

data_dir = Path("../output")
training_dir = Path("../data")

classification_path = data_dir / "MLC_classification.tif"

building_path = training_dir / "building_footprints.shp"

mask_path = data_dir / "Asbestos_mask.tif"

clean_mask_path = data_dir / "Asbestos_mask_clean.tif"

output_buildings = data_dir / "Buildings_AC.shp"



# User parameters
# Integer value corresponding to the asbestos class in the classified raster
asbestos_class = 1

# Median filter kernel size (must be odd)
median_kernel = 3

# Minimum object size (pixels)
minimum_patch_size = 9

# Minimum percentage of asbestos pixels required
threshold_percent = 30


# Read classified raster and extract the asbestos class
# Open the classified raster
with rasterio.open(classification_path) as src:

    classified = src.read(1)
    profile = src.profile.copy()

# Create a binary mask:
#   1 = asbestos class
#   0 = all other classes
asbestos_mask = (classified == asbestos_class).astype(np.uint8)



# Save binary asbestos mask
profile.update(
    dtype="uint8",
    count=1,
    nodata=0,
)

with rasterio.open(mask_path, "w", **profile) as dst:
    dst.write(asbestos_mask, 1)

print(f"Binary asbestos mask saved as:\n{mask_path.name}")



# Apply median filter
# Smoothing: reduce salt-and-pepper noise while preserving the shape of larger regions
smoothed_mask = cv2.medianBlur(
    asbestos_mask,
    ksize=median_kernel,
)



# Remove small isolated regions
# Remove connected pixels smaller than the selected minimum size
clean_mask = remove_small_objects(
    smoothed_mask.astype(bool),
    min_size=minimum_patch_size,
    connectivity=2,
)

clean_mask = clean_mask.astype(np.uint8)



# Save cleaned asbestos mask

with rasterio.open(clean_mask_path, "w", **profile) as dst:
    dst.write(clean_mask, 1)

print(f"Cleaned asbestos mask saved as:\n{clean_mask_path.name}")

In [ ]:
# Load cleaned asbestos mask

with rasterio.open(clean_mask_path) as src:

    asbestos_mask = src.read(1)
    transform = src.transform
    crs = src.crs

    # Pixel area (map units²)
    pixel_area = abs(src.transform.a * src.transform.e)



# Convert asbestos pixels to points
# Identify pixels classified as asbestos
rows, cols = np.where(asbestos_mask == 1)

# Convert raster indices to map coordinates
points = [
    Point(*xy(transform, row, col))
    for row, col in zip(rows, cols)
]

asbestos_pixels = gpd.GeoDataFrame(
    geometry=points,
    crs=crs,
)

print(f"Asbestos pixels extracted: {len(asbestos_pixels)}")



# Load building footprints

buildings = gpd.read_file(building_path).to_crs(crs)



# Assign asbestos pixels to buildings

joined = gpd.sjoin(
    asbestos_pixels,
    buildings,
    predicate="within",
)



# Calculate asbestos coverage for each building
# Count asbestos pixels per building
pixel_count = joined.groupby("index_right").size()

# Estimate the total number of pixels within each building
buildings["area_m2"] = buildings.area
buildings["total_pixels"] = buildings["area_m2"] / pixel_area

# Assign the number of asbestos pixels to each building
buildings["asbestos_pixels"] = (
    buildings.index
    .map(pixel_count)
    .fillna(0)
)

# Calculate asbestos coverage percentage
buildings["asbestos_percent"] = (
    buildings["asbestos_pixels"]
    / buildings["total_pixels"]
) * 100



# Select buildings exceeding the threshold
asbestos_buildings = buildings[
    buildings["asbestos_percent"] >= threshold_percent
]

print(
    f"Buildings with ≥{threshold_percent}% asbestos coverage: "
    f"{len(asbestos_buildings)}"
)



# Save output

asbestos_buildings.to_file(output_buildings)

print(f"Output saved as:\n{output_buildings.name}")